# Phase 4: Feature Selection

Selecting important features

In [ ]:
"""Phase 4: Feature SelectionThis script performs feature selection using correlation analysis and Random Forest importance.IMPORTANT: Uses model-based feature importance for final selection."""import osimport sysimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.ensemble import RandomForestClassifier# Set style for plotssns.set_style("whitegrid")plt.rcParams['figure.figsize'] = (12, 8)def load_balanced_data(filepath):    """Load the balanced dataset from previous step."""    print(f"\n{'='*60}")    print("STEP 1: Loading Balanced Data")    print(f"{'='*60}")        if not os.path.exists(filepath):        print(f"ERROR: File not found at {filepath}")        print("Please run pipeline/3_imbalance.py first to generate balanced_data.csv")        sys.exit(1)        try:        df = pd.read_csv(filepath)        print(f"✓ Successfully loaded data from {filepath}")        print(f"\nDataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")        print(f"\nColumn Names: {list(df.columns)}")                # Verify FTR exists        if 'FTR' not in df.columns:            print(f"\nERROR: Target column 'FTR' not found in dataset")            print(f"Available columns: {list(df.columns)}")            sys.exit(1)                print(f"\n✓ Target column 'FTR' verified")        print(f"\n✓ Step 1 completed successfully")        return df    except Exception as e:        print(f"ERROR: Failed to load CSV file: {e}")        sys.exit(1)def correlation_analysis(df, output_dir):    """Analyze correlation between features and target (exploratory only)."""    print(f"\n{'='*60}")    print("STEP 2: Correlation Analysis (Exploratory)")    print(f"{'='*60}")        # Select only numeric columns    numeric_df = df.select_dtypes(include=[np.number])        if numeric_df.empty:        print(f"\n⚠ WARNING: No numeric columns found. Skipping correlation analysis.")        return None        print(f"\nNumeric Columns Found: {list(numeric_df.columns)}")        if 'FTR' not in numeric_df.columns:        print(f"\n⚠ WARNING: FTR is not numeric. Cannot compute correlations.")        return None        # Compute correlation with FTR    correlations = numeric_df.corr()['FTR'].drop('FTR').sort_values(ascending=False)        print(f"\n{'='*50}")    print("IMPORTANT NOTE:")    print(f"{'='*50}")    print("Correlation with encoded FTR (0,1,2) is only INDICATIVE,")    print("not definitive for feature selection.")    print("FTR is categorical (Away Win, Draw, Home Win), not ordinal.")    print("Use Random Forest feature importance for final selection.")    print(f"{'='*50}")        print(f"\nFeature Correlations with FTR (sorted by absolute value):")    print(f"{'='*50}")        # Sort by absolute value for display    correlations_abs_sorted = correlations.abs().sort_values(ascending=False)        for i, feature in enumerate(correlations_abs_sorted.index, 1):        corr_value = correlations[feature]        abs_corr = abs(corr_value)        print(f"  {i:2d}. {feature:25s}: {corr_value:7.4f} (|{abs_corr:.4f}|)")        # Create bar chart    plt.figure(figsize=(12, 8))        # Plot top 15 features by absolute correlation    top_n = min(15, len(correlations))    top_features = correlations_abs_sorted.head(top_n).index    top_values = [correlations[feat] for feat in top_features]        colors = ['#2ecc71' if val > 0 else '#e74c3c' for val in top_values]        plt.barh(range(len(top_features)), top_values, color=colors, alpha=0.7, edgecolor='black')    plt.yticks(range(len(top_features)), top_features)    plt.xlabel('Correlation with FTR', fontsize=12)    plt.ylabel('Features', fontsize=12)    plt.title('Feature Correlations with FTR (Exploratory Only)', fontsize=14, fontweight='bold')    plt.axvline(x=0, color='black', linestyle='-', linewidth=0.8)    plt.grid(axis='x', alpha=0.3)    plt.tight_layout()        output_path = os.path.join(output_dir, 'feature_correlations.png')    plt.savefig(output_path, dpi=300, bbox_inches='tight')    plt.close()        print(f"\n✓ Chart saved: {output_path}")    print(f"\n✓ Step 2 completed successfully")        return correlationsdef prepare_data_for_selection(df):    """Prepare features (X) and target (y) for model-based selection."""    print(f"\n{'='*60}")    print("STEP 3: Preparing Data for Model-Based Selection")    print(f"{'='*60}")        # Select only numeric columns    numeric_df = df.select_dtypes(include=[np.number])        if numeric_df.empty:        print(f"\nERROR: No numeric columns found in dataset")        sys.exit(1)        # Separate features and target    if 'FTR' not in numeric_df.columns:        print(f"\nERROR: Target column 'FTR' not found in numeric columns")        sys.exit(1)        feature_columns = [col for col in numeric_df.columns if col != 'FTR']        if len(feature_columns) < 2:        print(f"\n⚠ WARNING: Only {len(feature_columns)} numeric feature(s) found")        print(f"Feature selection requires at least 2 features")        if len(feature_columns) == 0:            print(f"\nERROR: No features available for selection")            sys.exit(1)        X = numeric_df[feature_columns].copy()    y = numeric_df['FTR'].copy()        print(f"\nFeature List ({len(feature_columns)} features):")    for i, col in enumerate(feature_columns, 1):        print(f"  {i:2d}. {col}")        print(f"\nShape of X (features): {X.shape}")    print(f"Shape of y (target):   {y.shape}")        # Check for any NaN or infinite values    if X.isnull().any().any():        print(f"\n⚠ WARNING: NaN values detected in features")        nan_cols = X.columns[X.isnull().any()].tolist()        print(f"Columns with NaN: {nan_cols}")        if np.isinf(X.values).any():        print(f"\n⚠ WARNING: Infinite values detected in features")        print(f"\n✓ Step 3 completed successfully")        return X, y, feature_columnsdef random_forest_feature_importance(X, y, feature_columns, output_dir):    """Use Random Forest to compute feature importance."""    print(f"\n{'='*60}")    print("STEP 4: Random Forest Feature Importance")    print(f"{'='*60}")        print(f"\nTraining Random Forest Classifier...")    print(f"  n_estimators: 100")    print(f"  random_state: 42")    print(f"  n_jobs: -1 (use all CPU cores)")        # Train Random Forest    rf = RandomForestClassifier(        n_estimators=100,        random_state=42,        n_jobs=-1,        max_depth=10  # Prevent overfitting on full dataset    )        rf.fit(X, y)        print(f"\n✓ Random Forest trained successfully")        # Extract feature importances    importances = rf.feature_importances_        # Create DataFrame for easy sorting    importance_df = pd.DataFrame({        'Feature': feature_columns,        'Importance': importances    }).sort_values('Importance', ascending=False)        print(f"\nFeature Importance Ranking:")    print(f"{'='*50}")    print(f"{'Rank':<6} {'Feature':<25} {'Importance':<12}")    print(f"{'='*50}")        for i, row in importance_df.iterrows():        rank = importance_df.index.get_loc(i) + 1        print(f"{rank:<6} {row['Feature']:<25} {row['Importance']:<12.6f}")        print(f"\nTop 10 Features:")    print(f"{'='*50}")    top_10 = importance_df.head(10)    for i, row in top_10.iterrows():        rank = importance_df.index.get_loc(i) + 1        print(f"  {rank:2d}. {row['Feature']:<25} (importance: {row['Importance']:.6f})")        # Create horizontal bar chart    plt.figure(figsize=(12, 10))        # Plot all features (or top 20 if too many)    n_features_to_plot = min(20, len(importance_df))    plot_data = importance_df.head(n_features_to_plot)        colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(plot_data)))        plt.barh(range(len(plot_data)), plot_data['Importance'], color=colors, alpha=0.8, edgecolor='black')    plt.yticks(range(len(plot_data)), plot_data['Feature'])    plt.xlabel('Feature Importance', fontsize=12)    plt.ylabel('Features', fontsize=12)    plt.title('Random Forest Feature Importance', fontsize=14, fontweight='bold')    plt.gca().invert_yaxis()  # Highest importance at top    plt.grid(axis='x', alpha=0.3)    plt.tight_layout()        output_path = os.path.join(output_dir, 'feature_importance.png')    plt.savefig(output_path, dpi=300, bbox_inches='tight')    plt.close()        print(f"\n✓ Chart saved: {output_path}")    print(f"\n✓ Step 4 completed successfully")        return importance_dfdef select_final_features(importance_df, output_dir):    """Select top N features based on importance ranking."""    print(f"\n{'='*60}")    print("STEP 5: Selecting Final Features")    print(f"{'='*60}")        # Select top 10 features    n_features_to_select = min(10, len(importance_df))        if n_features_to_select < 10:        print(f"\n⚠ NOTE: Only {n_features_to_select} features available (less than 10)")        selected_features = importance_df.head(n_features_to_select)['Feature'].tolist()        print(f"\nSelected Features (Top {n_features_to_select}):")    print(f"{'='*50}")    for i, feature in enumerate(selected_features, 1):        importance = importance_df[importance_df['Feature'] == feature]['Importance'].values[0]        print(f"  {i:2d}. {feature:<25} (importance: {importance:.6f})")        # Save to text file    output_path = os.path.join(output_dir, 'selected_features.txt')    with open(output_path, 'w') as f:        for feature in selected_features:            f.write(f"{feature}\n")        print(f"\n✓ Selected features saved to: {output_path}")    print(f"\n✓ Step 5 completed successfully")        return selected_featuresdef save_feature_selected_data(df, selected_features, output_dir):    """Save dataset with only selected features + target."""    print(f"\n{'='*60}")    print("STEP 6: Saving Feature-Selected Dataset")    print(f"{'='*60}")        # Create new dataframe with selected features + FTR    columns_to_keep = selected_features + ['FTR']        # Verify all columns exist    missing_cols = [col for col in columns_to_keep if col not in df.columns]    if missing_cols:        print(f"\n⚠ WARNING: Some columns not found in dataset: {missing_cols}")        columns_to_keep = [col for col in columns_to_keep if col in df.columns]        df_selected = df[columns_to_keep].copy()        # Save to CSV    output_path = os.path.join(output_dir, 'feature_selected_data.csv')    df_selected.to_csv(output_path, index=False)        print(f"\n✓ Feature-selected dataset saved to: {output_path}")    print(f"\nFinal Dataset Shape: {df_selected.shape[0]} rows, {df_selected.shape[1]} columns")    print(f"\nFinal Column List:")    for i, col in enumerate(df_selected.columns, 1):        print(f"  {i:2d}. {col}")        print(f"\n✓ Step 6 completed successfully")print("\n" + "="*60)print("EPL MATCH DATA - FEATURE SELECTION")print("="*60)    # Define pathsinput_path = "pipeline/output/balanced_data.csv"output_dir = "pipeline/output"    # Ensure output directory existsos.makedirs(output_dir, exist_ok=True)    # Step 1: Load balanced datadf = load_balanced_data(input_path)    # Step 2: Correlation analysis (exploratory)correlations = correlation_analysis(df, output_dir)    # Step 3: Prepare data for model-based selectionX, y, feature_columns = prepare_data_for_selection(df)    # Step 4: Random Forest feature importanceimportance_df = random_forest_feature_importance(X, y, feature_columns, output_dir)    # Step 5: Select final featuresselected_features = select_final_features(importance_df, output_dir)    # Step 6: Save feature-selected datasetsave_feature_selected_data(df, selected_features, output_dir)    print(f"\n{'='*60}")print("✓ ALL FEATURE SELECTION STEPS COMPLETED!")print(f"{'='*60}")print(f"\nOutput Files Generated:")print(f"  1. {output_dir}/feature_correlations.png")print(f"  2. {output_dir}/feature_importance.png")print(f"  3. {output_dir}/selected_features.txt")print(f"  4. {output_dir}/feature_selected_data.csv")print(f"\nTop {len(selected_features)} features selected based on Random Forest importance")print(f"\nYou can now proceed to Phase 5: Model Training")print("="*60 + "\n")